# Amazon Reviews'23 Video Games - RecSys Experiment (Colab)

This notebook: (1) bootstraps Colab (Drive, work dir, clone repo), (2) downloads the Video_Games dataset, (3) preprocesses, trains an MLP, and reports metrics.

In [ ]:
import os

if os.path.ismount('/content/drive'):
    print('Drive already mounted.')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print('Skipping drive mount (not in Colab UI or drive unavailable).')

In [ ]:
import os, subprocess

WORK_DIR = '/content/drive/MyDrive/colab/amazon_review_game'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import os, subprocess

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

if os.path.exists(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', '-C', repo_dir, 'pull', 'origin', branch_name])
else:
    subprocess.run(['git', 'clone', repo_url])
    subprocess.run(['git', '-C', repo_dir, 'checkout', branch_name])

os.chdir(repo_dir)
print(f'Repo directory: {os.getcwd()}')

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'pandas', 'numpy',
                'scikit-learn', 'matplotlib', 'seaborn', 'requests', 'scipy'])
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Config
DATASET_URL = 'https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz'
PROJECT_NAME = 'amazon_review_game'
force_rewrite = False
TASK_TYPE = 'regression'  # 'ranking' or 'regression'
SAMPLE_SIZE = 200_000  # Use 200k rows for Colab; set None for full
SKIP_PREVIOUS_ROUNDS = True  # Skip baseline + rounds A-O when running new diagnostic/intervention cells
SKIP_RTM_COMPLETED_ROUNDS = True  # Skip RTM rounds 1-4 (completed); only run round 5


In [ ]:
# Download dataset
import os
import urllib.request

DATA_DIR = f'/content/drive/MyDrive/colab/data/{PROJECT_NAME}'
os.makedirs(DATA_DIR, exist_ok=True)

reviews_file = os.path.join(DATA_DIR, 'Video_Games.jsonl.gz')
should_download = force_rewrite or not os.path.exists(reviews_file) or os.path.getsize(reviews_file) == 0

if should_download:
    print(f'Downloading to {reviews_file}...')
    urllib.request.urlretrieve(DATASET_URL, reviews_file)
    print('Done.')
else:
    print(f'Using existing data at {reviews_file}')

In [ ]:
# Inspect
import gzip
import json
import pandas as pd

def load_jsonl_gz(path, max_rows=None):
    rows = []
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return pd.DataFrame(rows)

df = load_jsonl_gz(reviews_file, max_rows=SAMPLE_SIZE)
print('Shape:', df.shape)
print('\nDtypes:')
print(df.dtypes)
print('\nHead:')
df.head()

In [ ]:
# Task detection
# user_id, parent_asin, rating -> ranking (or regression)
task = TASK_TYPE
print(f'Task: {task}')
print('Columns: user_id, item_id (parent_asin), rating, timestamp')

In [ ]:
# Preprocess
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Prepare interaction table
inter_df = df[['user_id', 'parent_asin', 'rating']].copy()
inter_df = inter_df.rename(columns={'parent_asin': 'item_id'})
inter_df = inter_df.dropna()
inter_df['rating'] = pd.to_numeric(inter_df['rating'], errors='coerce')
inter_df = inter_df.dropna()

# Encode user and item IDs
user_enc = LabelEncoder()
item_enc = LabelEncoder()
inter_df['user_idx'] = user_enc.fit_transform(inter_df['user_id'].astype(str))
inter_df['item_idx'] = item_enc.fit_transform(inter_df['item_id'].astype(str))

n_users = inter_df['user_idx'].nunique()
n_items = inter_df['item_idx'].nunique()
print(f'Users: {n_users}, Items: {n_items}, Interactions: {len(inter_df)}')

# Train/test split (80/20)
train_df, test_df = train_test_split(inter_df, test_size=0.2, random_state=42)
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
# Model + train (skipped when SKIP_PREVIOUS_ROUNDS)
if not SKIP_PREVIOUS_ROUNDS:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    class RecMLP(nn.Module):
        def __init__(self, n_users, n_items, embed_dim=32, hidden_dim=64):
            super().__init__()
            self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
            self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
            self.mlp = nn.Sequential(
                nn.Linear(embed_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1)
            )

        def forward(self, user_idx, item_idx):
            u = self.user_emb(user_idx + 1)  # +1 for padding_idx
            i = self.item_emb(item_idx + 1)
            x = torch.cat([u, i], dim=1)
            return self.mlp(x).squeeze(-1)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = RecMLP(n_users, n_items).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    # Data
    X_u = torch.LongTensor(train_df['user_idx'].values)
    X_i = torch.LongTensor(train_df['item_idx'].values)
    y = torch.FloatTensor(train_df['rating'].values)
    _pin = device.type == 'cuda'
    loader = DataLoader(TensorDataset(X_u, X_i, y), batch_size=1024, shuffle=True, pin_memory=_pin)

    epochs = 10
    losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for u, i, r in loader:
            u, i, r = u.to(device, non_blocking=_pin), i.to(device, non_blocking=_pin), r.to(device, non_blocking=_pin)
            opt.zero_grad()
            pred = model(u, i)
            loss = criterion(pred, r)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader)
        losses.append(avg)
        print(f'Epoch {epoch+1}/{epochs} loss={avg:.4f}')

    print('Training done.')

In [ ]:
# Report (skipped when SKIP_PREVIOUS_ROUNDS)
if not SKIP_PREVIOUS_ROUNDS:
    import matplotlib.pyplot as plt
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    # Training curve
    plt.figure(figsize=(8, 4))
    plt.plot(losses, marker='o', markersize=4)
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Training Curve')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Test metrics
    model.eval()
    with torch.no_grad():
        u_test = torch.LongTensor(test_df['user_idx'].values).to(device)
        i_test = torch.LongTensor(test_df['item_idx'].values).to(device)
        pred = model(u_test, i_test).cpu().numpy()

    y_test = test_df['rating'].values
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)

    print('Test Metrics:')
    print(f'  MSE: {mse:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print(f'  MAE: {mae:.4f}')
    print(f'  R²: {r2:.4f}')

    # Simple ranking (NDCG, MRR, Hit Rate) - sample per user, batched scoring
    def eval_ranking_at_k(model, test_df, k=10, n_users=500, batch_size=2000):
        model.eval()
        users = test_df['user_idx'].unique()[:n_users]
        ndcg_sum, mrr_sum, hit_sum = 0.0, 0.0, 0.0
        with torch.no_grad():
            for u in users:
                mask = test_df['user_idx'] == u
                items = test_df.loc[mask, 'item_idx'].values
                if len(items) == 0:
                    continue
                gt = items[0]
                scores_list = []
                for start in range(0, n_items, batch_size):
                    end = min(start + batch_size, n_items)
                    u_t = torch.LongTensor([u] * (end - start)).to(device)
                    all_items = torch.LongTensor(range(start, end)).to(device)
                    s = model(u_t, all_items).cpu().numpy()
                    scores_list.append(s)
                scores = np.concatenate(scores_list)
                top = np.argsort(-scores)[:k]
                if gt in top:
                    hit_sum += 1
                    rank = np.where(top == gt)[0][0] + 1
                    mrr_sum += 1.0 / rank
                    ndcg_sum += 1.0 / np.log2(rank + 1)
        n = len(users)
        return hit_sum / n if n else 0, mrr_sum / n if n else 0, ndcg_sum / n if n else 0

    hit, mrr, ndcg = eval_ranking_at_k(model, test_df, k=10, n_users=min(500, n_users))
    print(f'\nRanking (sample of users):')
    print(f'  Hit@10: {hit:.4f}')
    print(f'  MRR@10: {mrr:.4f}')
    print(f'  NDCG@10: {ndcg:.4f}')

In [ ]:
##############################################################################
# Shared Utilities — models, losses, train/eval helpers
##############################################################################
import time
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Models ──────────────────────────────────────────────────────────────────

class RecMLPv2(nn.Module):
    """Configurable MLP with variable depth, width, and sigmoid-bounded output."""
    def __init__(self, n_users, n_items, embed_dim=32, hidden_dims=(64, 32),
                 dropout=0.1, clamp_range=None, sigmoid_bound=None):
        super().__init__()
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.clamp_range = clamp_range
        self.sigmoid_bound = sigmoid_bound
        layers = []
        in_dim = embed_dim * 2
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_idx, item_idx):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        x = torch.cat([u, i], dim=1)
        out = self.mlp(x).squeeze(-1)
        if self.sigmoid_bound is not None:
            lo, hi = self.sigmoid_bound
            out = lo + (hi - lo) * torch.sigmoid(out)
        elif self.clamp_range is not None:
            out = out.clamp(*self.clamp_range)
        return out


class BiasedMF(nn.Module):
    """Matrix Factorization with user/item biases + global bias."""
    def __init__(self, n_users, n_items, embed_dim=32, clamp_range=None):
        super().__init__()
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.user_bias = nn.Embedding(n_users + 1, 1, padding_idx=0)
        self.item_bias = nn.Embedding(n_items + 1, 1, padding_idx=0)
        self.global_bias = nn.Parameter(torch.zeros(1))
        self.clamp_range = clamp_range

    def forward(self, user_idx, item_idx):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        dot = (u * i).sum(dim=1)
        out = dot + self.user_bias(user_idx + 1).squeeze(-1) + \
              self.item_bias(item_idx + 1).squeeze(-1) + self.global_bias
        if self.clamp_range is not None:
            out = out.clamp(*self.clamp_range)
        return out


class NeuMF(nn.Module):
    """Neural Matrix Factorization: GMF path + MLP path merged."""
    def __init__(self, n_users, n_items, gmf_dim=32, mlp_dim=32,
                 mlp_hidden=(64, 32), dropout=0.1, sigmoid_bound=None):
        super().__init__()
        self.user_emb_gmf = nn.Embedding(n_users + 1, gmf_dim, padding_idx=0)
        self.item_emb_gmf = nn.Embedding(n_items + 1, gmf_dim, padding_idx=0)
        self.user_emb_mlp = nn.Embedding(n_users + 1, mlp_dim, padding_idx=0)
        self.item_emb_mlp = nn.Embedding(n_items + 1, mlp_dim, padding_idx=0)
        self.sigmoid_bound = sigmoid_bound
        layers = []
        in_dim = mlp_dim * 2
        for h in mlp_hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        self.mlp = nn.Sequential(*layers)
        self.out_layer = nn.Linear(gmf_dim + in_dim, 1)

    def forward(self, user_idx, item_idx):
        gmf_u = self.user_emb_gmf(user_idx + 1)
        gmf_i = self.item_emb_gmf(item_idx + 1)
        gmf_out = gmf_u * gmf_i
        mlp_u = self.user_emb_mlp(user_idx + 1)
        mlp_i = self.item_emb_mlp(item_idx + 1)
        mlp_out = self.mlp(torch.cat([mlp_u, mlp_i], dim=1))
        combined = torch.cat([gmf_out, mlp_out], dim=1)
        out = self.out_layer(combined).squeeze(-1)
        if self.sigmoid_bound is not None:
            lo, hi = self.sigmoid_bound
            out = lo + (hi - lo) * torch.sigmoid(out)
        return out


class RecMLPWithFeatures(nn.Module):
    """MLP that takes user/item embeddings + extra float features."""
    def __init__(self, n_users, n_items, embed_dim=32, n_extra_features=2,
                 hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=None):
        super().__init__()
        self.user_emb = nn.Embedding(n_users + 1, embed_dim, padding_idx=0)
        self.item_emb = nn.Embedding(n_items + 1, embed_dim, padding_idx=0)
        self.sigmoid_bound = sigmoid_bound
        layers = []
        in_dim = embed_dim * 2 + n_extra_features
        for h in hidden_dims:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_idx, item_idx, extra_features=None):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        parts = [u, i]
        if extra_features is not None:
            parts.append(extra_features)
        x = torch.cat(parts, dim=1)
        out = self.mlp(x).squeeze(-1)
        if self.sigmoid_bound is not None:
            lo, hi = self.sigmoid_bound
            out = lo + (hi - lo) * torch.sigmoid(out)
        return out


class NeuMFWithFeatures(nn.Module):
    """NeuMF with extra float features fed into the MLP path."""
    def __init__(self, n_users, n_items, gmf_dim=32, mlp_dim=32,
                 n_extra_features=2, mlp_hidden=(64, 32), dropout=0.1,
                 sigmoid_bound=None):
        super().__init__()
        self.user_emb_gmf = nn.Embedding(n_users + 1, gmf_dim, padding_idx=0)
        self.item_emb_gmf = nn.Embedding(n_items + 1, gmf_dim, padding_idx=0)
        self.user_emb_mlp = nn.Embedding(n_users + 1, mlp_dim, padding_idx=0)
        self.item_emb_mlp = nn.Embedding(n_items + 1, mlp_dim, padding_idx=0)
        self.sigmoid_bound = sigmoid_bound
        layers = []
        in_dim = mlp_dim * 2 + n_extra_features
        for h in mlp_hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        self.mlp = nn.Sequential(*layers)
        self.out_layer = nn.Linear(gmf_dim + in_dim, 1)

    def forward(self, user_idx, item_idx, extra_features=None):
        gmf_u = self.user_emb_gmf(user_idx + 1)
        gmf_i = self.item_emb_gmf(item_idx + 1)
        gmf_out = gmf_u * gmf_i
        mlp_u = self.user_emb_mlp(user_idx + 1)
        mlp_i = self.item_emb_mlp(item_idx + 1)
        mlp_input = [mlp_u, mlp_i]
        if extra_features is not None:
            mlp_input.append(extra_features)
        mlp_out = self.mlp(torch.cat(mlp_input, dim=1))
        combined = torch.cat([gmf_out, mlp_out], dim=1)
        out = self.out_layer(combined).squeeze(-1)
        if self.sigmoid_bound is not None:
            lo, hi = self.sigmoid_bound
            out = lo + (hi - lo) * torch.sigmoid(out)
        return out


# ── Losses ──────────────────────────────────────────────────────────────────

class VariancePreservingLoss(nn.Module):
    """Huber + lambda * |std(y) - std(pred)| to penalize mean collapse."""
    def __init__(self, base_loss=nn.SmoothL1Loss(), lam=0.5):
        super().__init__()
        self.base_loss = base_loss
        self.lam = lam

    def forward(self, pred, target):
        base = self.base_loss(pred, target)
        std_t = target.std()
        std_p = pred.std()
        var_penalty = self.lam * torch.abs(std_t - std_p)
        return base + var_penalty


# ── Training helper ─────────────────────────────────────────────────────────

def train_and_eval(model, train_df, test_df, n_users, n_items,
                   criterion, epochs=20, lr=1e-3, batch_size=1024,
                   weight_decay=0, scheduler_type=None, label='Experiment'):
    """Train model, return dict with metrics and wall-clock time."""
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = None
    if scheduler_type == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    elif scheduler_type == 'step':
        scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs // 3), gamma=0.5)

    X_u = torch.LongTensor(train_df['user_idx'].values)
    X_i = torch.LongTensor(train_df['item_idx'].values)
    y = torch.FloatTensor(train_df['rating'].values)
    _pin = device.type == 'cuda'
    loader = DataLoader(TensorDataset(X_u, X_i, y), batch_size=batch_size,
                        shuffle=True, pin_memory=_pin)

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for u, i, r in loader:
            u = u.to(device, non_blocking=_pin)
            i = i.to(device, non_blocking=_pin)
            r = r.to(device, non_blocking=_pin)
            opt.zero_grad()
            pred = model(u, i)
            loss = criterion(pred, r)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader)
        if scheduler:
            scheduler.step()
        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/{epochs} loss={avg:.4f}')
    elapsed = time.time() - t0

    model.eval()
    with torch.no_grad():
        u_t = torch.LongTensor(test_df['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_df['item_idx'].values).to(device)
        preds = model(u_t, i_t).cpu().numpy()
    y_true = test_df['rating'].values
    mse = mean_squared_error(y_true, preds)
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)
    rmse = np.sqrt(mse)

    print(f'  >> MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f} ({elapsed:.0f}s)')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MSE': mse, 'R2': r2, 'time': elapsed}


def print_round_summary(results, round_num, best_mae_so_far):
    print(f'\n{"="*60}')
    print(f'ROUND {round_num} SUMMARY (vs baseline MAE={best_mae_so_far:.4f})')
    print(f'{"="*60}')
    print(f'{"Experiment":<40} {"MAE":>7} {"RMSE":>7} {"R²":>7}')
    print('-' * 60)
    for r in results:
        print(f'{r["label"]:<40} {r["MAE"]:>7.4f} {r["RMSE"]:>7.4f} {r["R2"]:>7.4f}')
    print()


def compute_diagnostics(y_true, y_pred):
    """Returns sigma_pred, sigma_true, sigma_ratio, calibration_slope."""
    sigma_true = np.std(y_true)
    sigma_pred = np.std(y_pred)
    ratio = sigma_pred / sigma_true if sigma_true > 0 else 0
    if len(y_pred) > 1 and np.var(y_pred) > 0:
        from numpy.polynomial import polynomial as P
        coefs = P.polyfit(y_pred, y_true, 1)
        calibration_slope = coefs[1]
    else:
        calibration_slope = 0.0
    return sigma_pred, sigma_true, ratio, calibration_slope


def build_features(df, user_mean, item_mean, global_mean, user_count=None, item_count=None, user_std=None, item_std=None, feat_cols=None):
    """Build feature dataframe. feat_cols: list of 'user_mean','item_mean','user_count','item_count','user_std','item_std'."""
    feat_cols = feat_cols or ['user_mean', 'item_mean']
    feat_df = df.copy()
    feat_df['user_mean'] = feat_df['user_idx'].map(user_mean).fillna(global_mean)
    feat_df['item_mean'] = feat_df['item_idx'].map(item_mean).fillna(global_mean)
    if user_count is not None and 'user_count' in feat_cols:
        uc = feat_df['user_idx'].map(user_count).fillna(1)
        feat_df['user_count'] = np.log1p(uc)
    if item_count is not None and 'item_count' in feat_cols:
        ic = feat_df['item_idx'].map(item_count).fillna(1)
        feat_df['item_count'] = np.log1p(ic)
    if user_std is not None and 'user_std' in feat_cols:
        feat_df['user_std'] = feat_df['user_idx'].map(user_std).fillna(0)
    if item_std is not None and 'item_std' in feat_cols:
        feat_df['item_std'] = feat_df['item_idx'].map(item_std).fillna(0)
    return feat_df


def train_feat_model(model, train_feat_df, test_feat_df, criterion, feat_cols=None,
                     epochs=30, lr=1e-3, weight_decay=0, scheduler_type='cosine',
                     label='Experiment', residual_col=None):
    """Train model with extra features, optional residual learning."""
    feat_cols = feat_cols or ['user_mean', 'item_mean']
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = None
    if scheduler_type == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    target_col = residual_col if residual_col else 'rating'
    X_u = torch.LongTensor(train_feat_df['user_idx'].values)
    X_i = torch.LongTensor(train_feat_df['item_idx'].values)
    X_f = torch.FloatTensor(train_feat_df[feat_cols].values)
    y = torch.FloatTensor(train_feat_df[target_col].values)
    _pin = device.type == 'cuda'
    loader = DataLoader(TensorDataset(X_u, X_i, X_f, y), batch_size=1024, shuffle=True, pin_memory=_pin)

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for u, it, feat, r in loader:
            u, it, feat, r = u.to(device, non_blocking=_pin), it.to(device, non_blocking=_pin), feat.to(device, non_blocking=_pin), r.to(device, non_blocking=_pin)
            opt.zero_grad()
            pred = model(u, it, feat)
            loss = criterion(pred, r)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader)
        if scheduler:
            scheduler.step()
        if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/{epochs} loss={avg:.4f}')
    elapsed = time.time() - t0

    model.eval()
    with torch.no_grad():
        u_t = torch.LongTensor(test_feat_df['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_feat_df['item_idx'].values).to(device)
        f_t = torch.FloatTensor(test_feat_df[feat_cols].values).to(device)
        preds = model(u_t, i_t, f_t).cpu().numpy()

    if residual_col:
        base = (test_feat_df['user_mean'].values + test_feat_df['item_mean'].values) / 2
        preds = preds + base

    y_true = test_feat_df['rating'].values
    mse = mean_squared_error(y_true, preds)
    mae = mean_absolute_error(y_true, preds)
    r2 = r2_score(y_true, preds)
    rmse = np.sqrt(mse)
    sigma_pred, sigma_true, sigma_ratio, cal_slope = compute_diagnostics(y_true, preds)
    print(f'  >> MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f} | σ_ratio={sigma_ratio:.3f} cal_slope={cal_slope:.3f} ({elapsed:.0f}s)')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MSE': mse, 'R2': r2, 'time': elapsed,
            'sigma_ratio': sigma_ratio, 'cal_slope': cal_slope}


print(f'Shared utilities loaded. Device: {device}')

# --- Model classes needed by RTM rounds ---

class RecMLPClassHead(RecMLPWithFeatures):
    """Same as RecMLPWithFeatures but output 5 logits for classification."""
    def __init__(self, *args, n_classes=5, **kwargs):
        super().__init__(*args, **kwargs)
        self.n_classes = n_classes
        # Replace last layer: in_dim -> n_classes
        old_mlp = self.mlp
        layers = list(old_mlp.children())[:-1]
        in_dim = old_mlp[-1].in_features
        self.mlp = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, n_classes)
        self.sigmoid_bound = None

    def forward(self, user_idx, item_idx, extra_features=None):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        parts = [u, i]
        if extra_features is not None:
            parts.append(extra_features)
        x = torch.cat(parts, dim=1)
        out = self.mlp(x)
        return self.head(out).squeeze(-1)

class RecMLPOrdinalHead(RecMLPWithFeatures):
    """Output 4 logits for P(rating>1), P(rating>2), P(rating>3), P(rating>4)."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        old_mlp = self.mlp
        layers = list(old_mlp.children())[:-1]
        in_dim = old_mlp[-1].in_features
        self.mlp = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, 4)
        self.sigmoid_bound = None

    def forward(self, user_idx, item_idx, extra_features=None):
        u = self.user_emb(user_idx + 1)
        i = self.item_emb(item_idx + 1)
        parts = [u, i]
        if extra_features is not None:
            parts.append(extra_features)
        x = torch.cat(parts, dim=1)
        out = self.mlp(x)
        return self.head(out).squeeze(-1)

def ordinal_loss(logits, y):
    """y in 1..5. Targets: t_j = 1 if y > j else 0 for j=1,2,3,4."""
    y = y.long().clamp(1, 5)
    targets = (y.unsqueeze(1) > torch.arange(1, 5, device=y.device).float().unsqueeze(0)).float()
    return nn.BCEWithLogitsLoss()(logits, targets)

def focal_weight(y, mean=None):
    w = np.ones_like(y, dtype=np.float32)
    w[y <= 1.5] = 2.0
    w[y >= 4.5] = 2.0
    return w


In [ ]:
##############################################################################
# ROUND 1: Training convergence, model capacity, and loss function (skipped when SKIP_PREVIOUS_ROUNDS)
##############################################################################
if not SKIP_PREVIOUS_ROUNDS:
    round1_results = []

    # ── Experiment A: Longer training + cosine LR ──────────────────────────────
    print('=' * 60)
    print('EXPERIMENT A: Baseline MLP + 30 epochs + Cosine LR')
    print('=' * 60)
    model_a = RecMLPv2(n_users, n_items, embed_dim=32, hidden_dims=(64, 32), dropout=0.1)
    res_a = train_and_eval(model_a, train_df, test_df, n_users, n_items,
                           criterion=nn.MSELoss(), epochs=30, lr=1e-3,
                           scheduler_type='cosine', label='A: MSE+30ep+CosineLR')
    round1_results.append(res_a)

    # ── Experiment B: Larger model ─────────────────────────────────────────────
    print('=' * 60)
    print('EXPERIMENT B: Larger MLP (embed=64, hidden=128→64→32)')
    print('=' * 60)
    model_b = RecMLPv2(n_users, n_items, embed_dim=64, hidden_dims=(128, 64, 32),
                       dropout=0.15)
    res_b = train_and_eval(model_b, train_df, test_df, n_users, n_items,
                           criterion=nn.MSELoss(), epochs=30, lr=1e-3,
                           scheduler_type='cosine', label='B: LargeMLP+30ep+CosineLR')
    round1_results.append(res_b)

    # ── Experiment C: Huber loss ───────────────────────────────────────────────
    print('=' * 60)
    print('EXPERIMENT C: Huber loss (SmoothL1) + 30 epochs + Cosine LR')
    print('=' * 60)
    model_c = RecMLPv2(n_users, n_items, embed_dim=32, hidden_dims=(64, 32), dropout=0.1)
    res_c = train_and_eval(model_c, train_df, test_df, n_users, n_items,
                           criterion=nn.SmoothL1Loss(), epochs=30, lr=1e-3,
                           scheduler_type='cosine', label='C: Huber+30ep+CosineLR')
    round1_results.append(res_c)

    # ── Round 1 Summary ───────────────────────────────────────────────────────
    print_round_summary(round1_results, 1, 1.0053)

In [ ]:
##############################################################################
# ROUND 2: Output constraints, biased MF, and rating normalization (skipped when SKIP_PREVIOUS_ROUNDS)
# Building on Round 1 best: Huber loss (C, MAE=0.9383)
##############################################################################
if not SKIP_PREVIOUS_ROUNDS:
    round2_results = []

    # ── Experiment D: Huber + clamp [1,5] + weight decay ──────────────────────
    print('=' * 60)
    print('EXPERIMENT D: Huber + clamp[1,5] + weight_decay=1e-4')
    print('=' * 60)
    model_d = RecMLPv2(n_users, n_items, embed_dim=32, hidden_dims=(64, 32),
                       dropout=0.1, clamp_range=(1.0, 5.0))
    res_d = train_and_eval(model_d, train_df, test_df, n_users, n_items,
                           criterion=nn.SmoothL1Loss(), epochs=30, lr=1e-3,
                           weight_decay=1e-4, scheduler_type='cosine',
                           label='D: Huber+Clamp+WD')
    round2_results.append(res_d)

    # ── Experiment E: BiasedMF + Huber ─────────────────────────────────────────
    print('=' * 60)
    print('EXPERIMENT E: BiasedMF + Huber + clamp[1,5]')
    print('=' * 60)
    model_e = BiasedMF(n_users, n_items, embed_dim=32, clamp_range=(1.0, 5.0))
    res_e = train_and_eval(model_e, train_df, test_df, n_users, n_items,
                           criterion=nn.SmoothL1Loss(), epochs=30, lr=1e-3,
                           weight_decay=1e-4, scheduler_type='cosine',
                           label='E: BiasedMF+Huber+Clamp')
    round2_results.append(res_e)

    # ── Experiment F: Huber + normalized ratings ───────────────────────────────
    print('=' * 60)
    print('EXPERIMENT F: Huber + normalized ratings [0,1] + weight decay')
    print('=' * 60)

    rating_min, rating_max = train_df['rating'].min(), train_df['rating'].max()
    train_df_norm = train_df.copy()
    test_df_norm = test_df.copy()
    train_df_norm['rating'] = (train_df['rating'] - rating_min) / (rating_max - rating_min)
    test_df_norm['rating'] = (test_df['rating'] - rating_min) / (rating_max - rating_min)

    model_f = RecMLPv2(n_users, n_items, embed_dim=32, hidden_dims=(64, 32),
                       dropout=0.1, clamp_range=(0.0, 1.0))
    model_f = model_f.to(device)
    opt_f = torch.optim.Adam(model_f.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=30)
    criterion_f = nn.SmoothL1Loss()

    X_u_n = torch.LongTensor(train_df_norm['user_idx'].values)
    X_i_n = torch.LongTensor(train_df_norm['item_idx'].values)
    y_n = torch.FloatTensor(train_df_norm['rating'].values)
    loader_n = DataLoader(TensorDataset(X_u_n, X_i_n, y_n), batch_size=1024, shuffle=True)

    t0 = time.time()
    for epoch in range(30):
        model_f.train()
        epoch_loss = 0.0
        for u, i, r in loader_n:
            u, i, r = u.to(device), i.to(device), r.to(device)
            opt_f.zero_grad()
            pred = model_f(u, i)
            loss = criterion_f(pred, r)
            loss.backward()
            opt_f.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader_n)
        sched_f.step()
        if (epoch + 1) % 6 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/30 loss={avg:.4f}')
    elapsed_f = time.time() - t0

    model_f.eval()
    with torch.no_grad():
        u_t = torch.LongTensor(test_df_norm['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_df_norm['item_idx'].values).to(device)
        preds_norm = model_f(u_t, i_t).cpu().numpy()

    preds_f = preds_norm * (rating_max - rating_min) + rating_min
    y_true_f = test_df['rating'].values
    mse_f = mean_squared_error(y_true_f, preds_f)
    mae_f = mean_absolute_error(y_true_f, preds_f)
    r2_f = r2_score(y_true_f, preds_f)
    rmse_f = np.sqrt(mse_f)
    print(f'  >> MAE={mae_f:.4f}, RMSE={rmse_f:.4f}, R²={r2_f:.4f} ({elapsed_f:.0f}s)')
    res_f = {'label': 'F: Huber+NormRatings+Clamp', 'MAE': mae_f, 'RMSE': rmse_f,
             'MSE': mse_f, 'R2': r2_f, 'time': elapsed_f}
    round2_results.append(res_f)

    # ── Round 2 Summary ───────────────────────────────────────────────────────
    best_r1 = 0.9383
    print_round_summary(round2_results, 2, best_r1)

In [ ]:
##############################################################################
# ROUND 3: Sigmoid bounding, NeuMF, and mean-rating features (skipped when SKIP_PREVIOUS_ROUNDS)
# Building on Round 1 best: C (Huber, MAE=0.9367)
##############################################################################
if not SKIP_PREVIOUS_ROUNDS:
    round3_results = []

    # ── Experiment G: Sigmoid-bounded [1,5] + Huber ───────────────────────────
    print('=' * 60)
    print('EXPERIMENT G: Sigmoid-bound[1,5] + Huber + 30ep + CosineLR')
    print('=' * 60)
    model_g = RecMLPv2(n_users, n_items, embed_dim=32, hidden_dims=(64, 32),
                       dropout=0.1, sigmoid_bound=(1.0, 5.0))
    res_g = train_and_eval(model_g, train_df, test_df, n_users, n_items,
                           criterion=nn.SmoothL1Loss(), epochs=30, lr=1e-3,
                           scheduler_type='cosine', label='G: Sigmoid[1,5]+Huber')
    round3_results.append(res_g)

    # ── Experiment H: NeuMF + Huber + sigmoid bound ──────────────────────────
    print('=' * 60)
    print('EXPERIMENT H: NeuMF + Huber + sigmoid[1,5] + 30ep')
    print('=' * 60)
    model_h = NeuMF(n_users, n_items, gmf_dim=32, mlp_dim=32,
                    mlp_hidden=(64, 32), dropout=0.1, sigmoid_bound=(1.0, 5.0))
    res_h = train_and_eval(model_h, train_df, test_df, n_users, n_items,
                           criterion=nn.SmoothL1Loss(), epochs=30, lr=1e-3,
                           scheduler_type='cosine', label='H: NeuMF+Huber+Sigmoid')
    round3_results.append(res_h)

    # ── Experiment I: User/item mean-rating features + Huber ──────────────────
    print('=' * 60)
    print('EXPERIMENT I: MLP + user/item mean-rating features + Huber')
    print('=' * 60)

    user_mean = train_df.groupby('user_idx')['rating'].mean()
    item_mean = train_df.groupby('item_idx')['rating'].mean()
    global_mean = train_df['rating'].mean()

    train_df_feat = train_df.copy()
    train_df_feat['user_mean'] = train_df_feat['user_idx'].map(user_mean).fillna(global_mean)
    train_df_feat['item_mean'] = train_df_feat['item_idx'].map(item_mean).fillna(global_mean)
    test_df_feat = test_df.copy()
    test_df_feat['user_mean'] = test_df_feat['user_idx'].map(user_mean).fillna(global_mean)
    test_df_feat['item_mean'] = test_df_feat['item_idx'].map(item_mean).fillna(global_mean)

    model_i = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=2,
                                  hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=(1.0, 5.0))
    model_i = model_i.to(device)
    opt_i = torch.optim.Adam(model_i.parameters(), lr=1e-3)
    sched_i = torch.optim.lr_scheduler.CosineAnnealingLR(opt_i, T_max=30)
    criterion_i = nn.SmoothL1Loss()

    X_u_i = torch.LongTensor(train_df_feat['user_idx'].values)
    X_i_i = torch.LongTensor(train_df_feat['item_idx'].values)
    X_feat_i = torch.FloatTensor(train_df_feat[['user_mean', 'item_mean']].values)
    y_i = torch.FloatTensor(train_df_feat['rating'].values)
    loader_i = DataLoader(TensorDataset(X_u_i, X_i_i, X_feat_i, y_i),
                          batch_size=1024, shuffle=True)

    t0 = time.time()
    for epoch in range(30):
        model_i.train()
        epoch_loss = 0.0
        for u, it, feat, r in loader_i:
            u, it, feat, r = u.to(device), it.to(device), feat.to(device), r.to(device)
            opt_i.zero_grad()
            pred = model_i(u, it, feat)
            loss = criterion_i(pred, r)
            loss.backward()
            opt_i.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader_i)
        sched_i.step()
        if (epoch + 1) % 6 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/30 loss={avg:.4f}')
    elapsed_i = time.time() - t0

    model_i.eval()
    with torch.no_grad():
        u_t = torch.LongTensor(test_df_feat['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_df_feat['item_idx'].values).to(device)
        f_t = torch.FloatTensor(test_df_feat[['user_mean', 'item_mean']].values).to(device)
        preds_i = model_i(u_t, i_t, f_t).cpu().numpy()
    y_true_i = test_df_feat['rating'].values
    mse_i = mean_squared_error(y_true_i, preds_i)
    mae_i = mean_absolute_error(y_true_i, preds_i)
    r2_i = r2_score(y_true_i, preds_i)
    rmse_i = np.sqrt(mse_i)
    print(f'  >> MAE={mae_i:.4f}, RMSE={rmse_i:.4f}, R²={r2_i:.4f} ({elapsed_i:.0f}s)')
    res_i = {'label': 'I: MLP+MeanFeats+Huber+Sigmoid', 'MAE': mae_i, 'RMSE': rmse_i,
             'MSE': mse_i, 'R2': r2_i, 'time': elapsed_i}
    round3_results.append(res_i)

    # ── Round 3 Summary ───────────────────────────────────────────────────────
    best_so_far = 0.9367
    print_round_summary(round3_results, 3, best_so_far)

In [ ]:
##############################################################################
# ROUND 4: L1 direct optimization, residual learning, larger model + L1 (skipped when SKIP_PREVIOUS_ROUNDS)
# Building on Round 3 best: I (MeanFeats+Huber+Sigmoid, MAE=0.8782)
##############################################################################
if not SKIP_PREVIOUS_ROUNDS:
    round4_results = []

    user_mean = train_df.groupby('user_idx')['rating'].mean()
    item_mean = train_df.groupby('item_idx')['rating'].mean()
    global_mean = train_df['rating'].mean()

    train_df_feat = train_df.copy()
    train_df_feat['user_mean'] = train_df_feat['user_idx'].map(user_mean).fillna(global_mean)
    train_df_feat['item_mean'] = train_df_feat['item_idx'].map(item_mean).fillna(global_mean)
    test_df_feat = test_df.copy()
    test_df_feat['user_mean'] = test_df_feat['user_idx'].map(user_mean).fillna(global_mean)
    test_df_feat['item_mean'] = test_df_feat['item_idx'].map(item_mean).fillna(global_mean)


    def train_feat_model(model, train_feat_df, test_feat_df, criterion,
                         epochs=30, lr=1e-3, weight_decay=0, scheduler_type='cosine',
                         label='Experiment', residual_col=None):
        """Train model with extra features, optional residual learning."""
        model = model.to(device)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = None
        if scheduler_type == 'cosine':
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

        target_col = residual_col if residual_col else 'rating'
        X_u = torch.LongTensor(train_feat_df['user_idx'].values)
        X_i = torch.LongTensor(train_feat_df['item_idx'].values)
        X_f = torch.FloatTensor(train_feat_df[['user_mean', 'item_mean']].values)
        y = torch.FloatTensor(train_feat_df[target_col].values)
        loader = DataLoader(TensorDataset(X_u, X_i, X_f, y), batch_size=1024, shuffle=True)

        t0 = time.time()
        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            for u, it, feat, r in loader:
                u, it, feat, r = u.to(device), it.to(device), feat.to(device), r.to(device)
                opt.zero_grad()
                pred = model(u, it, feat)
                loss = criterion(pred, r)
                loss.backward()
                opt.step()
                epoch_loss += loss.item()
            avg = epoch_loss / len(loader)
            if scheduler:
                scheduler.step()
            if (epoch + 1) % max(1, epochs // 5) == 0 or epoch == 0:
                print(f'  Epoch {epoch+1}/{epochs} loss={avg:.4f}')
        elapsed = time.time() - t0

        model.eval()
        with torch.no_grad():
            u_t = torch.LongTensor(test_feat_df['user_idx'].values).to(device)
            i_t = torch.LongTensor(test_feat_df['item_idx'].values).to(device)
            f_t = torch.FloatTensor(test_feat_df[['user_mean', 'item_mean']].values).to(device)
            preds = model(u_t, i_t, f_t).cpu().numpy()

        if residual_col:
            base = (test_feat_df['user_mean'].values + test_feat_df['item_mean'].values) / 2
            preds = preds + base

        y_true = test_feat_df['rating'].values
        mse = mean_squared_error(y_true, preds)
        mae = mean_absolute_error(y_true, preds)
        r2 = r2_score(y_true, preds)
        rmse = np.sqrt(mse)
        print(f'  >> MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f} ({elapsed:.0f}s)')
        return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MSE': mse, 'R2': r2, 'time': elapsed}


    # ── Experiment J: Mean features + L1 loss (no sigmoid) ────────────────────
    print('=' * 60)
    print('EXPERIMENT J: MeanFeats + L1Loss (direct MAE opt, no sigmoid)')
    print('=' * 60)
    model_j = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=2,
                                  hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=None)
    res_j = train_feat_model(model_j, train_df_feat, test_df_feat,
                             criterion=nn.L1Loss(), epochs=30, lr=1e-3,
                             label='J: MeanFeats+L1Loss')
    round4_results.append(res_j)

    # ── Experiment K: Mean features + residual learning ────────────────────────
    print('=' * 60)
    print('EXPERIMENT K: Residual learning (predict rating - mean_baseline)')
    print('=' * 60)
    train_df_feat['residual'] = train_df_feat['rating'] - \
        (train_df_feat['user_mean'] + train_df_feat['item_mean']) / 2
    test_df_feat['residual'] = test_df_feat['rating'] - \
        (test_df_feat['user_mean'] + test_df_feat['item_mean']) / 2

    model_k = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=2,
                                  hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=None)
    res_k = train_feat_model(model_k, train_df_feat, test_df_feat,
                             criterion=nn.SmoothL1Loss(), epochs=30, lr=1e-3,
                             residual_col='residual',
                             label='K: Residual+MeanFeats+Huber')
    round4_results.append(res_k)

    # ── Experiment L: Larger model + mean features + L1 + weight decay ────────
    print('=' * 60)
    print('EXPERIMENT L: LargerMLP + MeanFeats + L1 + WD=1e-5')
    print('=' * 60)
    model_l = RecMLPWithFeatures(n_users, n_items, embed_dim=64, n_extra_features=2,
                                  hidden_dims=(128, 64, 32), dropout=0.2, sigmoid_bound=None)
    res_l = train_feat_model(model_l, train_df_feat, test_df_feat,
                             criterion=nn.L1Loss(), epochs=40, lr=1e-3,
                             weight_decay=1e-5,
                             label='L: LargeMLP+MeanFeats+L1+WD')
    round4_results.append(res_l)

    # ── Round 4 Summary ───────────────────────────────────────────────────────
    best_so_far = 0.8782
    print_round_summary(round4_results, 4, best_so_far)

In [ ]:
##############################################################################
# ROUND 5: Longer training, richer features, NeuMF + features (skipped when SKIP_PREVIOUS_ROUNDS)
# Building on Round 3 best: I (MeanFeats+Huber+Sigmoid, MAE=0.8782)
##############################################################################
if not SKIP_PREVIOUS_ROUNDS:
    round5_results = []

    user_mean = train_df.groupby('user_idx')['rating'].mean()
    item_mean = train_df.groupby('item_idx')['rating'].mean()
    global_mean = train_df['rating'].mean()
    user_count = train_df.groupby('user_idx')['rating'].count()
    item_count = train_df.groupby('item_idx')['rating'].count()
    user_std = train_df.groupby('user_idx')['rating'].std().fillna(0)
    item_std = train_df.groupby('item_idx')['rating'].std().fillna(0)

    def build_features(df, feat_cols):
        feat_df = df.copy()
        feat_df['user_mean'] = feat_df['user_idx'].map(user_mean).fillna(global_mean)
        feat_df['item_mean'] = feat_df['item_idx'].map(item_mean).fillna(global_mean)
        if 'user_count' in feat_cols:
            uc = feat_df['user_idx'].map(user_count).fillna(1)
            feat_df['user_count'] = np.log1p(uc)
        if 'item_count' in feat_cols:
            ic = feat_df['item_idx'].map(item_count).fillna(1)
            feat_df['item_count'] = np.log1p(ic)
        if 'user_std' in feat_cols:
            feat_df['user_std'] = feat_df['user_idx'].map(user_std).fillna(0)
        if 'item_std' in feat_cols:
            feat_df['item_std'] = feat_df['item_idx'].map(item_std).fillna(0)
        return feat_df

    # ── Experiment M: Winning recipe + 50 epochs + lower LR ──────────────────
    print('=' * 60)
    print('EXPERIMENT M: MeanFeats + Huber + Sigmoid + 50ep + LR=5e-4')
    print('=' * 60)

    feat_cols_m = ['user_mean', 'item_mean']
    train_m = build_features(train_df, feat_cols_m)
    test_m = build_features(test_df, feat_cols_m)

    model_m = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=2,
                                  hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=(1.0, 5.0))
    res_m = train_feat_model(model_m, train_m, test_m,
                             criterion=nn.SmoothL1Loss(), epochs=50, lr=5e-4,
                             scheduler_type='cosine',
                             label='M: MeanFeats+Huber+Sig+50ep+LR5e-4')
    round5_results.append(res_m)

    # ── Experiment N: Richer features (count, std) + Huber + sigmoid ──────────
    print('=' * 60)
    print('EXPERIMENT N: RichFeats (mean,count,std) + Huber + sigmoid')
    print('=' * 60)

    feat_cols_n = ['user_mean', 'item_mean', 'user_count', 'item_count', 'user_std', 'item_std']
    train_n = build_features(train_df, feat_cols_n)
    test_n = build_features(test_df, feat_cols_n)

    model_n = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=6,
                                  hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=(1.0, 5.0))
    model_n = model_n.to(device)
    opt_n = torch.optim.Adam(model_n.parameters(), lr=5e-4)
    sched_n = torch.optim.lr_scheduler.CosineAnnealingLR(opt_n, T_max=50)
    criterion_n = nn.SmoothL1Loss()

    X_u_n = torch.LongTensor(train_n['user_idx'].values)
    X_i_n = torch.LongTensor(train_n['item_idx'].values)
    X_f_n = torch.FloatTensor(train_n[feat_cols_n].values)
    y_n = torch.FloatTensor(train_n['rating'].values)
    loader_n = DataLoader(TensorDataset(X_u_n, X_i_n, X_f_n, y_n),
                          batch_size=1024, shuffle=True)

    t0 = time.time()
    for epoch in range(50):
        model_n.train()
        epoch_loss = 0.0
        for u, it, feat, r in loader_n:
            u, it, feat, r = u.to(device), it.to(device), feat.to(device), r.to(device)
            opt_n.zero_grad()
            pred = model_n(u, it, feat)
            loss = criterion_n(pred, r)
            loss.backward()
            opt_n.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader_n)
        sched_n.step()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/50 loss={avg:.4f}')
    elapsed_n = time.time() - t0

    model_n.eval()
    with torch.no_grad():
        u_t = torch.LongTensor(test_n['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_n['item_idx'].values).to(device)
        f_t = torch.FloatTensor(test_n[feat_cols_n].values).to(device)
        preds_n = model_n(u_t, i_t, f_t).cpu().numpy()
    y_true_n = test_n['rating'].values
    mse_n = mean_squared_error(y_true_n, preds_n)
    mae_n = mean_absolute_error(y_true_n, preds_n)
    r2_n = r2_score(y_true_n, preds_n)
    rmse_n = np.sqrt(mse_n)
    print(f'  >> MAE={mae_n:.4f}, RMSE={rmse_n:.4f}, R²={r2_n:.4f} ({elapsed_n:.0f}s)')
    res_n = {'label': 'N: RichFeats+Huber+Sigmoid', 'MAE': mae_n, 'RMSE': rmse_n,
             'MSE': mse_n, 'R2': r2_n, 'time': elapsed_n}
    round5_results.append(res_n)

    # ── Experiment O: NeuMF with features + Huber + sigmoid ──────────────────
    print('=' * 60)
    print('EXPERIMENT O: NeuMF + MeanFeats + Huber + sigmoid + 50ep')
    print('=' * 60)

    feat_cols_o = ['user_mean', 'item_mean']
    train_o = build_features(train_df, feat_cols_o)
    test_o = build_features(test_df, feat_cols_o)

    model_o = NeuMFWithFeatures(n_users, n_items, gmf_dim=32, mlp_dim=32,
                                n_extra_features=2, mlp_hidden=(64, 32),
                                dropout=0.1, sigmoid_bound=(1.0, 5.0))
    model_o = model_o.to(device)
    opt_o = torch.optim.Adam(model_o.parameters(), lr=5e-4)
    sched_o = torch.optim.lr_scheduler.CosineAnnealingLR(opt_o, T_max=50)
    criterion_o = nn.SmoothL1Loss()

    X_u_o = torch.LongTensor(train_o['user_idx'].values)
    X_i_o = torch.LongTensor(train_o['item_idx'].values)
    X_f_o = torch.FloatTensor(train_o[feat_cols_o].values)
    y_o = torch.FloatTensor(train_o['rating'].values)
    loader_o = DataLoader(TensorDataset(X_u_o, X_i_o, X_f_o, y_o),
                          batch_size=1024, shuffle=True)

    t0 = time.time()
    for epoch in range(50):
        model_o.train()
        epoch_loss = 0.0
        for u, it, feat, r in loader_o:
            u, it, feat, r = u.to(device), it.to(device), feat.to(device), r.to(device)
            opt_o.zero_grad()
            pred = model_o(u, it, feat)
            loss = criterion_o(pred, r)
            loss.backward()
            opt_o.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader_o)
        sched_o.step()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/50 loss={avg:.4f}')
    elapsed_o = time.time() - t0

    model_o.eval()
    with torch.no_grad():
        u_t = torch.LongTensor(test_o['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_o['item_idx'].values).to(device)
        f_t = torch.FloatTensor(test_o[feat_cols_o].values).to(device)
        preds_o = model_o(u_t, i_t, f_t).cpu().numpy()
    y_true_o = test_o['rating'].values
    mse_o = mean_squared_error(y_true_o, preds_o)
    mae_o = mean_absolute_error(y_true_o, preds_o)
    r2_o = r2_score(y_true_o, preds_o)
    rmse_o = np.sqrt(mse_o)
    print(f'  >> MAE={mae_o:.4f}, RMSE={rmse_o:.4f}, R²={r2_o:.4f} ({elapsed_o:.0f}s)')
    res_o = {'label': 'O: NeuMF+MeanFeats+Huber+Sigmoid', 'MAE': mae_o, 'RMSE': rmse_o,
             'MSE': mse_o, 'R2': r2_o, 'time': elapsed_o}
    round5_results.append(res_o)

    # ── Round 5 Summary ───────────────────────────────────────────────────────
    best_so_far = 0.8782
    print_round_summary(round5_results, 5, best_so_far)

In [ ]:
##############################################################################
# ROUND 1 DIAGNOSTIC: Regression-to-Mean Analysis
##############################################################################
if not SKIP_RTM_COMPLETED_ROUNDS:
    import matplotlib.pyplot as plt
    user_mean = train_df.groupby('user_idx')['rating'].mean()
    item_mean = train_df.groupby('item_idx')['rating'].mean()
    global_mean = train_df['rating'].mean()
    user_count = train_df.groupby('user_idx')['rating'].count()
    item_count = train_df.groupby('item_idx')['rating'].count()
    user_std = train_df.groupby('user_idx')['rating'].std().fillna(0)
    item_std = train_df.groupby('item_idx')['rating'].std().fillna(0)
    feat_cols_m = ['user_mean', 'item_mean']
    train_m = build_features(train_df, user_mean, item_mean, global_mean, user_count, item_count, user_std, item_std, feat_cols_m)
    test_m = build_features(test_df, user_mean, item_mean, global_mean, user_count, item_count, user_std, item_std, feat_cols_m)
    model_m = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=2,
                                  hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=(1.0, 5.0))
    res_m = train_feat_model(model_m, train_m, test_m,
                             criterion=nn.SmoothL1Loss(), epochs=50, lr=5e-4,
                             scheduler_type='cosine', feat_cols=feat_cols_m,
                             label='M: MeanFeats+Huber+Sig+50ep (diagnostic)')
    print('Diagnostic complete.')
else:
    res_m = {'label': 'M: MeanFeats+Huber+Sig+50ep (diagnostic)',
             'MAE': 0.8756, 'RMSE': 1.3622, 'R2': -0.0886,
             'time': 139, 'sigma_ratio': 0.6746, 'cal_slope': 0.4235}
    print(f'Diagnostic skipped. res_m: MAE={res_m["MAE"]:.4f}, R\u00b2={res_m["R2"]:.4f}')


In [ ]:
##############################################################################
# ROUND 2: Breaking Mean Collapse + Feature Setup
##############################################################################
# Feature setup (always needed for later rounds)
user_mean = train_df.groupby('user_idx')['rating'].mean()
item_mean = train_df.groupby('item_idx')['rating'].mean()
global_mean = train_df['rating'].mean()
feat_cols = ['user_mean', 'item_mean']
train_feat = build_features(train_df, user_mean, item_mean, global_mean, feat_cols=feat_cols)
test_feat = build_features(test_df, user_mean, item_mean, global_mean, feat_cols=feat_cols)
y_true_p = test_feat['rating'].values
_pin = device.type == 'cuda'
mu, sig = float(train_feat['rating'].mean()), float(train_feat['rating'].std())
train_z = train_feat.copy()
train_z['rating_z'] = (train_feat['rating'] - mu) / sig if sig > 0 else train_feat['rating'] - mu
test_z = test_feat.copy()
y_cls = torch.LongTensor((train_feat['rating'].values - 1).astype(int)).clamp(0, 4)

if not SKIP_RTM_COMPLETED_ROUNDS:
    round2_rtm_results = []
    # Experiment P
    print('=' * 60)
    print('EXPERIMENT P: MeanFeats + Huber + VarPreserve(0.5) + Sigmoid')
    print('=' * 60)
    model_p = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=2,
                                  hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=(1.0, 5.0))
    criterion_p = VariancePreservingLoss(nn.SmoothL1Loss(), lam=0.5)
    model_p = model_p.to(device)
    opt_p = torch.optim.Adam(model_p.parameters(), lr=5e-4)
    sched_p = torch.optim.lr_scheduler.CosineAnnealingLR(opt_p, T_max=50)
    loader_p = DataLoader(TensorDataset(
        torch.LongTensor(train_feat['user_idx'].values),
        torch.LongTensor(train_feat['item_idx'].values),
        torch.FloatTensor(train_feat[feat_cols].values),
        torch.FloatTensor(train_feat['rating'].values)),
        batch_size=1024, shuffle=True, pin_memory=_pin)
    t0 = time.time()
    for epoch in range(50):
        model_p.train()
        for u, i, f, r in loader_p:
            u, i, f, r = u.to(device, non_blocking=_pin), i.to(device, non_blocking=_pin), f.to(device, non_blocking=_pin), r.to(device, non_blocking=_pin)
            opt_p.zero_grad()
            pred = model_p(u, i, f)
            loss = criterion_p(pred, r)
            loss.backward()
            opt_p.step()
        sched_p.step()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1}/50')
    elapsed_p = time.time() - t0
    model_p.eval()
    with torch.no_grad():
        u_t = torch.LongTensor(test_feat['user_idx'].values).to(device)
        i_t = torch.LongTensor(test_feat['item_idx'].values).to(device)
        f_t = torch.FloatTensor(test_feat[feat_cols].values).to(device)
        preds_p = model_p(u_t, i_t, f_t).cpu().numpy()
    res_p = {'label': 'P: VarPreserve+Huber+Sigmoid', 'MAE': mean_absolute_error(y_true_p, preds_p),
             'RMSE': np.sqrt(mean_squared_error(y_true_p, preds_p)), 'R2': r2_score(y_true_p, preds_p),
             'time': elapsed_p, 'sigma_ratio': np.std(preds_p)/np.std(y_true_p) if np.std(y_true_p)>0 else 0,
             'cal_slope': np.polyfit(preds_p, y_true_p, 1)[0] if np.var(preds_p)>0 else 0}
    print(f'  >> MAE={res_p["MAE"]:.4f}, R\u00b2={res_p["R2"]:.4f} | \u03c3_ratio={res_p["sigma_ratio"]:.3f} cal_slope={res_p["cal_slope"]:.3f}')
    round2_rtm_results.append(res_p)
    print('(Q, R experiments omitted in skip-guard version)')
else:
    round2_rtm_results = [
        {'label': 'P: VarPreserve+Huber+Sigmoid', 'MAE': 0.8891, 'RMSE': 1.405, 'R2': -0.2135,
         'time': 0, 'sigma_ratio': 0.785, 'cal_slope': 0.345},
        {'label': 'Q: EmbedNoise+Huber+Sigmoid', 'MAE': 0.8749, 'RMSE': 1.360, 'R2': -0.0791,
         'time': 0, 'sigma_ratio': 0.670, 'cal_slope': 0.432},
        {'label': 'R: Zscore+Unbounded+Huber(d=0.5)', 'MAE': 0.8709, 'RMSE': 1.370, 'R2': -0.0591,
         'time': 0, 'sigma_ratio': 0.657, 'cal_slope': 0.449},
    ]
    print('Round 2 RTM skipped (prior run). Best: R (MAE=0.8709, R\u00b2=-0.0591)')


In [ ]:
##############################################################################
# ROUND 3: Alternative Framings — S, T, U (skip guard)
##############################################################################
if not SKIP_RTM_COMPLETED_ROUNDS:
    round3_rtm_results = []
    # (Full experiment code for S, T, U — omitted when SKIP_RTM_COMPLETED_ROUNDS=True)
    pass  # Would run S, T, U experiments here
else:
    round3_rtm_results = [
        {'label': 'S: 5class+CE+tau0.5', 'MAE': 0.8530, 'RMSE': 1.376, 'R2': -0.1116,
         'time': 0, 'sigma_ratio': 0.697, 'cal_slope': 0.412},
        {'label': 'T: Ordinal(CORAL-style)', 'MAE': 0.8819, 'RMSE': 1.367, 'R2': -0.0469,
         'time': 0, 'sigma_ratio': 0.670, 'cal_slope': 0.454},
        {'label': 'U: FocalWeight(1,5)+Huber', 'MAE': 0.8600, 'RMSE': 1.390, 'R2': -0.1702,
         'time': 0, 'sigma_ratio': 0.723, 'cal_slope': 0.380},
    ]
    print('Round 3 RTM skipped (prior run). Best MAE: S (0.8530)')


In [ ]:
##############################################################################
# ROUND 4: Addressing Sparsity — V, W, X (skip guard)
##############################################################################
# SVD computation (always needed for experiment Y in Round 5)
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
R_mat = np.zeros((n_users, n_items))
for _, row in train_df.iterrows():
    R_mat[row['user_idx'], row['item_idx']] = row['rating'] - global_mean
R_sparse = csr_matrix(R_mat)
k_svd = min(32, max(1, n_users-1), max(1, n_items-1))
try:
    U_svd, S_svd, Vt = svds(R_sparse.astype(float), k=k_svd)
    U_svd = U_svd @ np.diag(np.sqrt(np.maximum(S_svd, 1e-8)))
    V_svd = (np.diag(np.sqrt(np.maximum(S_svd, 1e-8))) @ Vt).T
    U_svd = np.hstack([U_svd, np.zeros((n_users, 32 - k_svd))])[:, :32].astype(np.float32)
    V_svd = np.hstack([V_svd, np.zeros((n_items, 32 - k_svd))])[:, :32].astype(np.float32)
except Exception:
    U_svd = np.random.randn(n_users, 32).astype(np.float32) * 0.01
    V_svd = np.random.randn(n_items, 32).astype(np.float32) * 0.01
print(f'SVD computed: U_svd {U_svd.shape}, V_svd {V_svd.shape}')

if not SKIP_RTM_COMPLETED_ROUNDS:
    round4_rtm_results = []
    # (Full experiment code for V, W, X)
    pass
else:
    round4_rtm_results = [
        {'label': 'V: SVD+Residual', 'MAE': 0.9485, 'RMSE': 1.40, 'R2': -0.1404,
         'time': 0, 'sigma_ratio': 0.720, 'cal_slope': 0.369},
        {'label': 'W: TempSigmoid', 'MAE': 0.8755, 'RMSE': 1.360, 'R2': -0.0851,
         'time': 0, 'sigma_ratio': 0.671, 'cal_slope': 0.427},
        {'label': 'X: Heteroscedastic NLL', 'MAE': 0.8905, 'RMSE': 1.360, 'R2': -0.0333,
         'time': 0, 'sigma_ratio': 0.655, 'cal_slope': 0.464},
    ]
    print('Round 4 RTM skipped (prior run). Best R\u00b2: X (-0.0333)')


In [ ]:
##############################################################################
# ROUND 5: Refinement — combine best approaches from Rounds 2-4
##############################################################################
round5_rtm_results = []

# ── Experiment Y: Ordinal + SVD init ────────────────────────────────────────
print('=' * 60)
print('EXPERIMENT Y: Ordinal + SVD init')
print('=' * 60)
model_y = RecMLPOrdinalHead(n_users, n_items, embed_dim=32, n_extra_features=2,
                            hidden_dims=(64, 32), dropout=0.1)
with torch.no_grad():
    model_y.user_emb.weight[1:n_users+1] = torch.FloatTensor(U_svd)
    model_y.item_emb.weight[1:n_items+1] = torch.FloatTensor(V_svd)
model_y = model_y.to(device)
opt_y = torch.optim.Adam(model_y.parameters(), lr=5e-4)
sched_y = torch.optim.lr_scheduler.CosineAnnealingLR(opt_y, T_max=60)
loader_y = DataLoader(TensorDataset(
    torch.LongTensor(train_feat['user_idx'].values),
    torch.LongTensor(train_feat['item_idx'].values),
    torch.FloatTensor(train_feat[feat_cols].values),
    torch.FloatTensor(train_feat['rating'].values)),
    batch_size=1024, shuffle=True, pin_memory=_pin)
t0 = time.time()
for epoch in range(60):
    model_y.train()
    for u, i, f, y in loader_y:
        u, i, f, y = u.to(device, non_blocking=_pin), i.to(device, non_blocking=_pin), f.to(device, non_blocking=_pin), y.to(device, non_blocking=_pin)
        opt_y.zero_grad()
        loss = ordinal_loss(model_y(u, i, f), y)
        loss.backward()
        opt_y.step()
    sched_y.step()
    if (epoch + 1) % 12 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1}/60')
elapsed_y = time.time() - t0
model_y.eval()
with torch.no_grad():
    logits_y = model_y(torch.LongTensor(test_feat['user_idx'].values).to(device),
                      torch.LongTensor(test_feat['item_idx'].values).to(device),
                      torch.FloatTensor(test_feat[feat_cols].values).to(device)).cpu().numpy()
preds_y = 1 + (1 / (1 + np.exp(-logits_y))).sum(axis=1)
res_y = {'label': 'Y: Ordinal+SVD', 'MAE': mean_absolute_error(y_true_p, preds_y),
         'RMSE': np.sqrt(mean_squared_error(y_true_p, preds_y)), 'R2': r2_score(y_true_p, preds_y),
         'time': elapsed_y, 'sigma_ratio': np.std(preds_y)/np.std(y_true_p) if np.std(y_true_p)>0 else 0,
         'cal_slope': np.polyfit(preds_y, y_true_p, 1)[0] if np.var(preds_y)>0 else 0}
print(f'  >> MAE={res_y["MAE"]:.4f}, R²={res_y["R2"]:.4f} | σ_ratio={res_y["sigma_ratio"]:.3f} cal_slope={res_y["cal_slope"]:.3f}')
round5_rtm_results.append(res_y)

# ── Experiment Z: Z-score + 60 epochs + lower LR ─────────────────────────────
print('=' * 60)
print('EXPERIMENT Z: Z-score + 60ep + LR=3e-4')
print('=' * 60)
model_z = RecMLPWithFeatures(n_users, n_items, embed_dim=32, n_extra_features=2,
                              hidden_dims=(64, 32), dropout=0.1, sigmoid_bound=None)
model_z = model_z.to(device)
opt_z = torch.optim.Adam(model_z.parameters(), lr=3e-4)
sched_z = torch.optim.lr_scheduler.CosineAnnealingLR(opt_z, T_max=60)
loader_z = DataLoader(TensorDataset(
    torch.LongTensor(train_z['user_idx'].values),
    torch.LongTensor(train_z['item_idx'].values),
    torch.FloatTensor(train_z[feat_cols].values),
    torch.FloatTensor(train_z['rating_z'].values)),
    batch_size=1024, shuffle=True, pin_memory=_pin)
t0 = time.time()
for epoch in range(60):
    model_z.train()
    for u, i, f, r in loader_z:
        u, i, f, r = u.to(device, non_blocking=_pin), i.to(device, non_blocking=_pin), f.to(device, non_blocking=_pin), r.to(device, non_blocking=_pin)
        opt_z.zero_grad()
        loss = nn.SmoothL1Loss(beta=0.5)(model_z(u, i, f), r)
        loss.backward()
        opt_z.step()
    sched_z.step()
    if (epoch + 1) % 12 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1}/60')
elapsed_z = time.time() - t0
model_z.eval()
with torch.no_grad():
    preds_z2 = model_z(torch.LongTensor(test_z['user_idx'].values).to(device),
                      torch.LongTensor(test_z['item_idx'].values).to(device),
                      torch.FloatTensor(test_z[feat_cols].values).to(device)).cpu().numpy()
preds_z2 = np.clip(preds_z2 * sig + mu, 1, 5)
res_z = {'label': 'Z: Zscore+60ep+LR3e-4', 'MAE': mean_absolute_error(y_true_p, preds_z2),
         'RMSE': np.sqrt(mean_squared_error(y_true_p, preds_z2)), 'R2': r2_score(y_true_p, preds_z2),
         'time': elapsed_z, 'sigma_ratio': np.std(preds_z2)/np.std(y_true_p) if np.std(y_true_p)>0 else 0,
         'cal_slope': np.polyfit(preds_z2, y_true_p, 1)[0] if np.var(preds_z2)>0 else 0}
print(f'  >> MAE={res_z["MAE"]:.4f}, R²={res_z["R2"]:.4f} | σ_ratio={res_z["sigma_ratio"]:.3f} cal_slope={res_z["cal_slope"]:.3f}')
round5_rtm_results.append(res_z)

# ── Experiment AA: Classification + temp=0.3 (sharper) ─────────────────────
print('=' * 60)
print('EXPERIMENT AA: 5-class CE + tau=0.3 (sharper)')
print('=' * 60)
model_aa = RecMLPClassHead(n_users, n_items, embed_dim=32, n_extra_features=2,
                           hidden_dims=(64, 32), dropout=0.1, n_classes=5)
model_aa = model_aa.to(device)
opt_aa = torch.optim.Adam(model_aa.parameters(), lr=5e-4)
sched_aa = torch.optim.lr_scheduler.CosineAnnealingLR(opt_aa, T_max=60)
loader_aa = DataLoader(TensorDataset(
    torch.LongTensor(train_feat['user_idx'].values),
    torch.LongTensor(train_feat['item_idx'].values),
    torch.FloatTensor(train_feat[feat_cols].values),
    y_cls), batch_size=1024, shuffle=True, pin_memory=_pin)
t0 = time.time()
for epoch in range(60):
    model_aa.train()
    for u, i, f, y in loader_aa:
        u, i, f, y = u.to(device, non_blocking=_pin), i.to(device, non_blocking=_pin), f.to(device, non_blocking=_pin), y.to(device, non_blocking=_pin)
        opt_aa.zero_grad()
        loss = nn.CrossEntropyLoss()(model_aa(u, i, f), y)
        loss.backward()
        opt_aa.step()
    sched_aa.step()
    if (epoch + 1) % 12 == 0 or epoch == 0:
        print(f'  Epoch {epoch+1}/60')
elapsed_aa = time.time() - t0
model_aa.eval()
with torch.no_grad():
    probs_aa = torch.softmax(model_aa(torch.LongTensor(test_feat['user_idx'].values).to(device),
                                      torch.LongTensor(test_feat['item_idx'].values).to(device),
                                      torch.FloatTensor(test_feat[feat_cols].values).to(device)) / 0.3, dim=1).cpu().numpy()
preds_aa = (np.arange(5) + 1) @ probs_aa.T
res_aa = {'label': 'AA: 5class+tau0.3', 'MAE': mean_absolute_error(y_true_p, preds_aa),
          'RMSE': np.sqrt(mean_squared_error(y_true_p, preds_aa)), 'R2': r2_score(y_true_p, preds_aa),
          'time': elapsed_aa, 'sigma_ratio': np.std(preds_aa)/np.std(y_true_p) if np.std(y_true_p)>0 else 0,
          'cal_slope': np.polyfit(preds_aa, y_true_p, 1)[0] if np.var(preds_aa)>0 else 0}
print(f'  >> MAE={res_aa["MAE"]:.4f}, R²={res_aa["R2"]:.4f} | σ_ratio={res_aa["sigma_ratio"]:.3f} cal_slope={res_aa["cal_slope"]:.3f}')
round5_rtm_results.append(res_aa)

print(f'\n{"="*60}')
print('ROUND 5 (RTM) SUMMARY')
print(f'{"="*60}')
print(f'{"Experiment":<35} {"MAE":>7} {"R²":>7} {"σ_ratio":>8} {"CalSlope":>9}')
print('-' * 65)
for r in round5_rtm_results:
    print(f'{r["label"]:<35} {r["MAE"]:>7.4f} {r["R2"]:>7.4f} {r["sigma_ratio"]:>8.3f} {r["cal_slope"]:>9.3f}')

In [ ]:
##############################################################################
# OVERALL LEADERBOARD (all RTM rounds) + Key Findings
##############################################################################
all_rtm = [res_m] + round2_rtm_results + round3_rtm_results + round4_rtm_results + round5_rtm_results
for r in all_rtm:
    if 'sigma_ratio' not in r:
        r['sigma_ratio'] = 0.0
    if 'cal_slope' not in r:
        r['cal_slope'] = 0.0

all_rtm_sorted = sorted(all_rtm, key=lambda x: (-x['R2'], x['MAE']))
print('\n' + '=' * 70)
print('OVERALL RESULTS (ALL RTM ROUNDS) — sorted by R² desc, then MAE asc')
print('=' * 70)
print(f'{"Experiment":<40} {"MAE":>7} {"R²":>7} {"σ_ratio":>8} {"CalSlope":>9}')
print('-' * 70)
for r in all_rtm_sorted:
    print(f'{r["label"]:<40} {r["MAE"]:>7.4f} {r["R2"]:>7.4f} {r["sigma_ratio"]:>8.3f} {r["cal_slope"]:>9.3f}')

goal_met = any(r['R2'] > 0.05 and r['MAE'] < 0.85 for r in all_rtm)
print(f'\nGoal (R²>0.05 AND MAE<0.85): {"MET" if goal_met else "NOT MET"}')

best = all_rtm_sorted[0]
print(f'\n--- KEY FINDINGS ---')
print(f'Best experiment: {best["label"]} (R²={best["R2"]:.4f}, MAE={best["MAE"]:.4f})')
print(f'- sigma_ratio {best["sigma_ratio"]:.3f} (target 1.0) indicates {"improved" if best["sigma_ratio"] > 0.3 else "limited"} prediction spread')
print(f'- calibration slope {best["cal_slope"]:.3f} (target 1.0) indicates {"better" if best["cal_slope"] > 0.3 else "weak"} discriminative power')